# The training loop, line by line

A print shop wants a turnaround estimate (in hours) from the page count, the paper type and the number of
copies. You will train a small neural network on its past jobs with the eight-line PyTorch loop, then break the
loop on purpose, one line at a time, and read what the output tells you. Same jobs and same code as the lesson.

About 5 minutes. No account, key or GPU needed. Run the cells from top to bottom
(Runtime > Run all works too). Exact numbers may differ a little on another PyTorch or NumPy version.

## 1. The print jobs

This cell makes the data. The jobs are simulated: a small script made them up from a known rule with some
random spread (seed 0), so every run gives the same jobs. The rule: 0.5 hours of setup, plus sheets
(pages × copies) divided by how many sheets per hour the press does on that paper, times a random factor
between 0.9 and 1.1.

`features` turns each job into 5 numbers for the model: pages/100, copies/100, and the paper type as three
0-or-1 columns (plain, glossy, card).

**Look for:** job 0 has 103 pages, 290 copies on card and took 18.40 hours. There are 400 jobs, from 0.50 to
23.41 hours.

In [1]:
import numpy as np

PAPERS = ["plain", "glossy", "card"]
SHEETS_PER_HOUR = {"plain": 6000, "glossy": 3000, "card": 1500}


def make_jobs(n=400, seed=0):
    rng = np.random.default_rng(seed)
    pages = rng.integers(1, 121, n)          # 1..120 pages
    copies = rng.integers(1, 301, n)         # 1..300 copies
    paper = rng.integers(0, 3, n)            # index into PAPERS
    rate = np.array([SHEETS_PER_HOUR[p] for p in PAPERS])[paper]
    hours = (0.5 + pages * copies / rate) * rng.uniform(0.9, 1.1, n)
    return pages, copies, paper, hours


def features(pages, copies, paper):
    """5 numbers per job: pages/100, copies/100, and paper type one-hot (plain, glossy, card)."""
    onehot = np.eye(3)[paper]
    return np.column_stack([pages / 100, copies / 100, onehot]).astype(np.float32)


pages, copies, paper, hours = make_jobs()
for i in range(5):
    print(f"job {i}: pages={pages[i]} copies={copies[i]} paper={PAPERS[paper[i]]} hours={hours[i]:.2f}")
print(f"jobs={len(hours)} hours min={hours.min():.2f} mean={hours.mean():.2f} max={hours.max():.2f}")
print("features of job 0:", features(pages, copies, paper)[0])

job 0: pages=103 copies=290 paper=card hours=18.40
job 1: pages=77 copies=96 paper=plain hours=1.81
job 2: pages=62 copies=240 paper=glossy hours=5.65
job 3: pages=33 copies=57 paper=card hours=1.81
job 4: pages=37 copies=109 paper=card hours=3.31
jobs=400 hours min=0.50 mean=4.18 max=23.41
features of job 0: [1.03 2.9  0.   0.   1.  ]


## 2. The loop from scratch, in NumPy

Before PyTorch, the loop in plain NumPy, as on the lesson's code screen. The model is a straight line:
hours = jobs @ weights + bias. We use only the first 6 jobs and two inputs (pages/100, copies/100),
`batch_size=4`, 2 epochs, learning rate 0.1. The gradient of the mean squared error is written by hand.

The `verbose` argument and its `print` are added so you can follow each batch; the rest is the lesson's code.

**Look for:** each epoch has 2 batches: one of 4 jobs, then one of only 2 (the jobs left over). In epoch 0 the
first batch is jobs [4, 0, 2, 1]. After 2 epochs the weights are [2.371, 5.077] and the bias is 1.45.

In [2]:
rng = np.random.default_rng(1)   # the shuffles


def train_numpy(jobs, hours, epochs, batch_size, lr, verbose=False):
    weights, bias = np.zeros(jobs.shape[1]), 0.0
    for epoch in range(epochs):
        order = rng.permutation(len(jobs))
        for start in range(0, len(jobs), batch_size):
            batch = order[start:start + batch_size]
            predicted = jobs[batch] @ weights + bias
            error = predicted - hours[batch]
            weights = weights - lr * 2 * jobs[batch].T @ error / len(batch)
            bias = bias - lr * 2 * error.mean()
            if verbose:
                print(f"epoch {epoch}  batch {batch.tolist()}  error {np.round(error, 2).tolist()}"
                      f"  -> weights {np.round(weights, 2).tolist()}  bias {bias:.2f}")
    return weights, bias


x = features(pages, copies, paper).astype(float)
small_jobs, small_hours = x[:6, :2], hours[:6]
print("small_jobs (pages/100, copies/100):", np.round(small_jobs, 2).tolist())
print("small_hours:", np.round(small_hours, 2).tolist())
w, b = train_numpy(small_jobs, small_hours, epochs=2, batch_size=4, lr=0.1, verbose=True)
print("result: weights", np.round(w, 3).tolist(), " bias", round(float(b), 3))

small_jobs (pages/100, copies/100): [[1.03, 2.9], [0.77, 0.96], [0.62, 2.4], [0.33, 0.57], [0.37, 1.09], [0.05, 2.02]]
small_hours: [18.4, 1.81, 5.65, 1.81, 3.31, 0.68]
epoch 0  batch [4, 0, 2, 1]  error [-3.31, -18.4, -5.65, -1.81]  -> weights [1.25, 3.61]  bias 1.46
epoch 0  batch [5, 3]  error [8.14, 2.12]  -> weights [1.14, 1.85]  bias 0.43
epoch 1  batch [2, 3, 5, 4]  error [-0.08, 0.05, 3.54, -0.44]  -> weights [1.14, 1.52]  bias 0.28
epoch 1  batch [0, 1]  error [-12.53, 0.81]  -> weights [2.37, 5.08]  bias 1.45
result: weights [2.371, 5.077]  bias 1.45


## 3. The NumPy loop on all the training jobs

Now the same function on 300 jobs with all 5 inputs, `batch_size=32`, learning rate 0.01. The other 100 jobs
are held back: the model never trains on them, we only measure on them. The loss is the mean squared error
(hours squared).

**Look for:** after 1 epoch the training loss is 15.460; after 30 epochs it is 5.827, and the loss on the
held-back jobs (6.299) is a little higher. A straight line cannot follow pages × copies well, so it stops here.

In [3]:
def mse(w, b, j, h):
    return float(np.mean((j @ w + b - h) ** 2))


for epochs in [1, 30]:
    w, b = train_numpy(x[:300], hours[:300], epochs, 32, 0.01)
    print(f"epochs {epochs:2d}  train loss {mse(w, b, x[:300], hours[:300]):.3f}  val loss {mse(w, b, x[300:], hours[300:]):.3f}")

epochs  1  train loss 15.460  val loss 15.155
epochs 30  train loss 5.827  val loss 6.299


## 4. The PyTorch setup

`setup` builds everything the loop needs, the same way for every run below: the 300 training jobs and the
100 held-back jobs as tensors, a small network (5 inputs, 16 hidden units, 1 output), the mean squared error
loss, plain SGD with learning rate 0.01, and a `DataLoader` that hands out shuffled batches of 32.
`torch.manual_seed(0)` is set first, so every run starts from the same weights and the same shuffles.

`after` prints two checks on the first layer when training ends: how far its weights moved from where they
started, and the size (norm) of its stored gradient.

**Look for:** 300 training jobs, 100 held back, and 10 batches per epoch (9 of 32 jobs and one of 12).

In [4]:
import torch
from torch import nn
from torch.utils.data import DataLoader, TensorDataset

EPOCHS = 30
LR = 0.01
BATCH = 32


def setup(batch_size=BATCH):
    pages, copies, paper, hours = make_jobs()
    x = torch.tensor(features(pages, copies, paper))
    y = torch.tensor(hours, dtype=torch.float32).unsqueeze(1)
    train_jobs, train_hours = x[:300], y[:300]       # 300 jobs to learn from
    val_jobs, val_hours = x[300:], y[300:]           # 100 jobs held back
    torch.manual_seed(0)
    model = nn.Sequential(nn.Linear(5, 16), nn.ReLU(), nn.Linear(16, 1))
    loss_fn = nn.MSELoss()
    optimizer = torch.optim.SGD(model.parameters(), lr=LR)
    train_loader = DataLoader(TensorDataset(train_jobs, train_hours), batch_size=batch_size, shuffle=True)
    return model, loss_fn, optimizer, train_loader, train_jobs, train_hours, val_jobs, val_hours


def report(epoch, train_loss, val_loss, val_mae):
    print(f"epoch {epoch:2d}  train loss {train_loss.item():9.3f}  val loss {val_loss.item():9.3f}  val off by {val_mae.item():6.2f} h")


def after(model, start_weight):
    w = model[0].weight
    change = (w.detach() - start_weight).abs().max().item()
    grad = "None" if w.grad is None else f"{w.grad.norm().item():.3f}"
    print(f"first layer: largest weight change {change:.6f}  grad norm {grad}")


print("torch", torch.__version__)
parts = setup()
train_loader = parts[3]
print("training jobs:", len(parts[4]), " held back:", len(parts[6]))
print("batches per epoch:", len(train_loader), " batch sizes:", [len(h) for _, h in train_loader])

torch 2.15.0


training jobs: 300  held back: 100
batches per epoch: 10  batch sizes: [32, 32, 32, 32, 32, 32, 32, 32, 32, 12]


## 5. The full training loop

This is `train_ok`, the loop from the lesson. For each epoch: training mode, then for each batch: guess,
measure the loss, clear the old gradients, work out new ones, move the weights. After each epoch, the
validation pass measures the model on the 100 held-back jobs inside `torch.no_grad()`, with no backward and
no step. "val off by" is the average error in hours on those jobs.

**Look for:** the training loss starts at 11.397 and ends at 1.441 after epoch 30. It does not fall every
epoch (epoch 13 is worse than epoch 12): judge the trend. The validation loss stays a little above the training
loss in every epoch, and ends at 1.753 (off by 0.92 hours). The loop made 300 weight updates.

In [5]:
def train_ok(model, loss_fn, optimizer, train_loader, train_jobs, train_hours, val_jobs, val_hours, epochs=EPOCHS):
    updates = 0
    for epoch in range(1, epochs + 1):
        model.train()
        for jobs, hours in train_loader:
            predicted = model(jobs)
            loss = loss_fn(predicted, hours)
            optimizer.zero_grad()
            loss.backward()
            optimizer.step()
            updates += 1
        model.eval()
        with torch.no_grad():
            train_loss = loss_fn(model(train_jobs), train_hours)
            val_loss = loss_fn(model(val_jobs), val_hours)
            val_mae = (model(val_jobs) - val_hours).abs().mean()
        report(epoch, train_loss, val_loss, val_mae)
    return updates


parts = setup()
model = parts[0]
start = model[0].weight.detach().clone()
print("weight updates:", train_ok(*parts))
after(model, start)

epoch  1  train loss    11.397  val loss    11.634  val off by   2.27 h
epoch  2  train loss     9.873  val loss    10.304  val off by   2.25 h
epoch  3  train loss     9.322  val loss     9.530  val off by   1.83 h
epoch  4  train loss     7.616  val loss     8.139  val off by   1.95 h
epoch  5  train loss     6.919  val loss     7.306  val off by   1.61 h
epoch  6  train loss     5.986  val loss     6.610  val off by   1.70 h
epoch  7  train loss     5.628  val loss     6.014  val off by   1.39 h
epoch  8  train loss     4.891  val loss     5.574  val off by   1.51 h
epoch  9  train loss     4.349  val loss     4.964  val off by   1.36 h
epoch 10  train loss     4.172  val loss     4.688  val off by   1.34 h
epoch 11  train loss     3.720  val loss     4.326  val off by   1.31 h
epoch 12  train loss     4.075  val loss     4.475  val off by   1.37 h
epoch 13  train loss     4.805  val loss     5.691  val off by   1.75 h
epoch 14  train loss     2.999  val loss     3.574  val off by  

## 6. Broken on purpose: no `optimizer.step()`

The same loop with one line deleted: `optimizer.step()`. Same seed, same data, same starting weights.
Nothing raises an error.

**Look for:** every epoch prints the same row (train loss 36.401, val loss 34.808). The last line says the
largest weight change is 0.000000, but the gradient norm is 10.324: `backward()` filled the gradients, and
nothing used them.

In [6]:
def train_no_step(model, loss_fn, optimizer, train_loader, train_jobs, train_hours, val_jobs, val_hours, epochs=EPOCHS):
    for epoch in range(1, epochs + 1):
        model.train()
        for jobs, hours in train_loader:
            predicted = model(jobs)
            loss = loss_fn(predicted, hours)
            optimizer.zero_grad()
            loss.backward()
            # optimizer.step()   <- deleted
        model.eval()
        with torch.no_grad():
            train_loss = loss_fn(model(train_jobs), train_hours)
            val_loss = loss_fn(model(val_jobs), val_hours)
            val_mae = (model(val_jobs) - val_hours).abs().mean()
        report(epoch, train_loss, val_loss, val_mae)


parts = setup()
model = parts[0]
start = model[0].weight.detach().clone()
train_no_step(*parts)
after(model, start)

epoch  1  train loss    36.401  val loss    34.808  val off by   4.19 h
epoch  2  train loss    36.401  val loss    34.808  val off by   4.19 h
epoch  3  train loss    36.401  val loss    34.808  val off by   4.19 h
epoch  4  train loss    36.401  val loss    34.808  val off by   4.19 h
epoch  5  train loss    36.401  val loss    34.808  val off by   4.19 h
epoch  6  train loss    36.401  val loss    34.808  val off by   4.19 h
epoch  7  train loss    36.401  val loss    34.808  val off by   4.19 h
epoch  8  train loss    36.401  val loss    34.808  val off by   4.19 h
epoch  9  train loss    36.401  val loss    34.808  val off by   4.19 h
epoch 10  train loss    36.401  val loss    34.808  val off by   4.19 h
epoch 11  train loss    36.401  val loss    34.808  val off by   4.19 h
epoch 12  train loss    36.401  val loss    34.808  val off by   4.19 h
epoch 13  train loss    36.401  val loss    34.808  val off by   4.19 h
epoch 14  train loss    36.401  val loss    34.808  val off by  

epoch 22  train loss    36.401  val loss    34.808  val off by   4.19 h
epoch 23  train loss    36.401  val loss    34.808  val off by   4.19 h
epoch 24  train loss    36.401  val loss    34.808  val off by   4.19 h
epoch 25  train loss    36.401  val loss    34.808  val off by   4.19 h
epoch 26  train loss    36.401  val loss    34.808  val off by   4.19 h
epoch 27  train loss    36.401  val loss    34.808  val off by   4.19 h
epoch 28  train loss    36.401  val loss    34.808  val off by   4.19 h
epoch 29  train loss    36.401  val loss    34.808  val off by   4.19 h
epoch 30  train loss    36.401  val loss    34.808  val off by   4.19 h
first layer: largest weight change 0.000000  grad norm 10.324


## 7. Broken on purpose: no `loss.backward()`

Now `loss.backward()` is deleted instead, and `step()` stays.

**Look for:** the very same flat rows as in step 6. The flat loss alone cannot tell the two bugs apart. The
last line can: the gradient norm is `None`, because no gradient was ever worked out, so `step()` had nothing
to move the weights with.

In [7]:
def train_no_backward(model, loss_fn, optimizer, train_loader, train_jobs, train_hours, val_jobs, val_hours, epochs=EPOCHS):
    for epoch in range(1, epochs + 1):
        model.train()
        for jobs, hours in train_loader:
            predicted = model(jobs)
            loss = loss_fn(predicted, hours)
            optimizer.zero_grad()
            # loss.backward()   <- deleted
            optimizer.step()
        model.eval()
        with torch.no_grad():
            train_loss = loss_fn(model(train_jobs), train_hours)
            val_loss = loss_fn(model(val_jobs), val_hours)
            val_mae = (model(val_jobs) - val_hours).abs().mean()
        report(epoch, train_loss, val_loss, val_mae)


parts = setup()
model = parts[0]
start = model[0].weight.detach().clone()
train_no_backward(*parts)
after(model, start)

epoch  1  train loss    36.401  val loss    34.808  val off by   4.19 h
epoch  2  train loss    36.401  val loss    34.808  val off by   4.19 h
epoch  3  train loss    36.401  val loss    34.808  val off by   4.19 h
epoch  4  train loss    36.401  val loss    34.808  val off by   4.19 h
epoch  5  train loss    36.401  val loss    34.808  val off by   4.19 h
epoch  6  train loss    36.401  val loss    34.808  val off by   4.19 h
epoch  7  train loss    36.401  val loss    34.808  val off by   4.19 h
epoch  8  train loss    36.401  val loss    34.808  val off by   4.19 h
epoch  9  train loss    36.401  val loss    34.808  val off by   4.19 h
epoch 10  train loss    36.401  val loss    34.808  val off by   4.19 h
epoch 11  train loss    36.401  val loss    34.808  val off by   4.19 h
epoch 12  train loss    36.401  val loss    34.808  val off by   4.19 h
epoch 13  train loss    36.401  val loss    34.808  val off by   4.19 h
epoch 14  train loss    36.401  val loss    34.808  val off by  

## 8. Broken on purpose: no `optimizer.zero_grad()`

Last broken copy: `zero_grad()` is deleted. PyTorch's `backward()` adds each new gradient to the one already
stored in `.grad`. Without clearing, every step uses the sum of all batches so far.

**Look for:** the loss falls for the first epochs (16.570, 9.188, 6.034, 2.848, 2.298), then swings and climbs
to 20.719 at epoch 30. The gradient norm at the end is 100.670, about 10 times the 9.792 of the full loop, and
the weights moved 96 instead of about 1.

In [8]:
def train_no_zero_grad(model, loss_fn, optimizer, train_loader, train_jobs, train_hours, val_jobs, val_hours, epochs=EPOCHS):
    for epoch in range(1, epochs + 1):
        model.train()
        for jobs, hours in train_loader:
            predicted = model(jobs)
            loss = loss_fn(predicted, hours)
            # optimizer.zero_grad()   <- deleted
            loss.backward()
            optimizer.step()
        model.eval()
        with torch.no_grad():
            train_loss = loss_fn(model(train_jobs), train_hours)
            val_loss = loss_fn(model(val_jobs), val_hours)
            val_mae = (model(val_jobs) - val_hours).abs().mean()
        report(epoch, train_loss, val_loss, val_mae)


parts = setup()
model = parts[0]
start = model[0].weight.detach().clone()
train_no_zero_grad(*parts)
after(model, start)

epoch  1  train loss    16.570  val loss    15.964  val off by   2.32 h
epoch  2  train loss     9.188  val loss     8.811  val off by   1.93 h
epoch  3  train loss     6.034  val loss     7.020  val off by   1.84 h
epoch  4  train loss     2.848  val loss     3.313  val off by   1.42 h
epoch  5  train loss     2.298  val loss     2.279  val off by   1.16 h
epoch  6  train loss     2.809  val loss     2.392  val off by   1.20 h
epoch  7  train loss     5.709  val loss     5.659  val off by   1.64 h
epoch  8  train loss     3.254  val loss     3.388  val off by   1.13 h
epoch  9  train loss     5.110  val loss     5.298  val off by   1.77 h
epoch 10  train loss     5.111  val loss     5.234  val off by   1.72 h
epoch 11  train loss     3.166  val loss     3.742  val off by   1.64 h
epoch 12  train loss     2.609  val loss     3.281  val off by   1.57 h
epoch 13  train loss     2.076  val loss     2.212  val off by   1.22 h
epoch 14  train loss    13.929  val loss    13.239  val off by  

epoch 16  train loss    15.250  val loss    15.385  val off by   2.89 h
epoch 17  train loss    15.025  val loss    15.920  val off by   3.32 h
epoch 18  train loss    13.131  val loss    11.536  val off by   2.81 h
epoch 19  train loss    19.176  val loss    14.166  val off by   2.21 h
epoch 20  train loss    24.057  val loss    22.410  val off by   3.29 h
epoch 21  train loss     9.482  val loss     7.689  val off by   1.95 h
epoch 22  train loss    19.129  val loss    18.057  val off by   3.31 h
epoch 23  train loss    21.880  val loss    22.512  val off by   4.11 h
epoch 24  train loss    17.721  val loss    17.366  val off by   2.90 h
epoch 25  train loss    22.005  val loss    20.998  val off by   2.73 h
epoch 26  train loss    17.951  val loss    17.542  val off by   2.82 h
epoch 27  train loss    21.571  val loss    22.366  val off by   4.08 h
epoch 28  train loss    18.982  val loss    19.092  val off by   3.53 h
epoch 29  train loss    20.602  val loss    19.334  val off by  

## 9. Batch size: how many updates per epoch?

The full loop again, with batch sizes 1, 32 and 300. One batch gives one weight update, so one epoch makes
300 / batch size updates (rounded up). We print only the first and the last epoch of each run.

**Look for:** 300, 10 and 1 updates per epoch (9000, 300 and 30 in total). After 30 epochs batch size 1 is
the closest (off by 0.43 hours) and batch size 300 the furthest (off by 2.08 hours). Batch size 1 takes the
longest to run: many small steps cost more trips round the loop.

In [9]:
import contextlib
import io
import time

print(f"seed 0  epochs {EPOCHS}  lr {LR}  300 training jobs")
for batch_size in [1, 32, 300]:
    parts = setup(batch_size)
    buf = io.StringIO()
    t0 = time.time()
    with contextlib.redirect_stdout(buf):   # keep only the first and last epoch rows
        updates = train_ok(*parts)
    lines = buf.getvalue().splitlines()
    print(f"batch_size {batch_size:3d}  updates per epoch {updates // EPOCHS:3d}  total updates {updates:4d}  ({time.time() - t0:.1f} s)")
    print("   " + lines[0])
    print("   " + lines[-1])

seed 0  epochs 30  lr 0.01  300 training jobs


batch_size   1  updates per epoch 300  total updates 9000  (1.8 s)
   epoch  1  train loss     3.192  val loss     3.895  val off by   1.24 h
   epoch 30  train loss     0.390  val loss     0.459  val off by   0.43 h
batch_size  32  updates per epoch  10  total updates  300  (0.1 s)
   epoch  1  train loss    11.397  val loss    11.634  val off by   2.27 h
   epoch 30  train loss     1.441  val loss     1.753  val off by   0.92 h
batch_size 300  updates per epoch   1  total updates   30  (0.0 s)
   epoch  1  train loss    31.466  val loss    30.106  val off by   3.69 h
   epoch 30  train loss     8.622  val loss     9.116  val off by   2.08 h


## Now change it

Three small experiments. Each cell has one line marked `# <- change`. Edit it, run the cell, and compare with
the full loop in step 5 (train loss 1.441 and off by 0.92 hours at epoch 30).

### Experiment 1: the learning rate

The learning rate sets how far `step()` moves the weights along each gradient. Change `LR_TRY` to `0.001`,
run, then try `0.03`.

**What should change:** with 0.001 each step is ten times smaller, so after 30 epochs the training loss is
still high (about 8.6, close to the batch-size-300 run). With 0.03 the steps are bigger and it ends far below
1.441 (about 0.5) with the same 300 updates. A rate that is too big can overshoot instead.

In [10]:
LR_TRY = 0.01  # <- change: try 0.001, then 0.03
model, loss_fn, optimizer, train_loader, *data = setup()
optimizer = torch.optim.SGD(model.parameters(), lr=LR_TRY)
print("learning rate", LR_TRY)
print("weight updates:", train_ok(model, loss_fn, optimizer, train_loader, *data))

learning rate 0.01
epoch  1  train loss    11.397  val loss    11.634  val off by   2.27 h
epoch  2  train loss     9.873  val loss    10.304  val off by   2.25 h
epoch  3  train loss     9.322  val loss     9.530  val off by   1.83 h
epoch  4  train loss     7.616  val loss     8.139  val off by   1.95 h
epoch  5  train loss     6.919  val loss     7.306  val off by   1.61 h
epoch  6  train loss     5.986  val loss     6.610  val off by   1.70 h
epoch  7  train loss     5.628  val loss     6.014  val off by   1.39 h
epoch  8  train loss     4.891  val loss     5.574  val off by   1.51 h
epoch  9  train loss     4.349  val loss     4.964  val off by   1.36 h
epoch 10  train loss     4.172  val loss     4.688  val off by   1.34 h
epoch 11  train loss     3.720  val loss     4.326  val off by   1.31 h
epoch 12  train loss     4.075  val loss     4.475  val off by   1.37 h
epoch 13  train loss     4.805  val loss     5.691  val off by   1.75 h
epoch 14  train loss     2.999  val loss     

epoch 25  train loss     1.640  val loss     1.995  val off by   0.94 h
epoch 26  train loss     1.577  val loss     1.912  val off by   0.92 h
epoch 27  train loss     1.817  val loss     2.064  val off by   0.94 h
epoch 28  train loss     1.438  val loss     1.735  val off by   0.88 h
epoch 29  train loss     1.409  val loss     1.721  val off by   0.90 h
epoch 30  train loss     1.441  val loss     1.753  val off by   0.92 h
weight updates: 300


### Experiment 2: drop the last, smaller batch

300 jobs in batches of 32 leave 12 jobs over, and the loader keeps them as a tenth, smaller batch. Change
`DROP_LAST` to `True`.

**What should change:** 9 batches per epoch instead of 10, so 270 weight updates instead of 300. Each epoch
now skips 12 jobs (different ones each time, because of the shuffle), and the numbers end a little different
from step 5.

In [11]:
DROP_LAST = False  # <- change: try True
model, loss_fn, optimizer, train_loader, *data = setup()
train_loader = DataLoader(train_loader.dataset, batch_size=BATCH, shuffle=True, drop_last=DROP_LAST)
print("batches per epoch:", len(train_loader))
print("weight updates:", train_ok(model, loss_fn, optimizer, train_loader, *data))

batches per epoch: 10
epoch  1  train loss    11.397  val loss    11.634  val off by   2.27 h
epoch  2  train loss     9.873  val loss    10.304  val off by   2.25 h
epoch  3  train loss     9.322  val loss     9.530  val off by   1.83 h
epoch  4  train loss     7.616  val loss     8.139  val off by   1.95 h
epoch  5  train loss     6.919  val loss     7.306  val off by   1.61 h
epoch  6  train loss     5.986  val loss     6.610  val off by   1.70 h
epoch  7  train loss     5.628  val loss     6.014  val off by   1.39 h
epoch  8  train loss     4.891  val loss     5.574  val off by   1.51 h
epoch  9  train loss     4.349  val loss     4.964  val off by   1.36 h
epoch 10  train loss     4.172  val loss     4.688  val off by   1.34 h
epoch 11  train loss     3.720  val loss     4.326  val off by   1.31 h
epoch 12  train loss     4.075  val loss     4.475  val off by   1.37 h
epoch 13  train loss     4.805  val loss     5.691  val off by   1.75 h
epoch 14  train loss     2.999  val loss  

### Experiment 3: `zero_grad()` in the wrong place

Here the loop is written out again with a switch for where `zero_grad()` runs. Change `ZERO_GRAD_EVERY` to
`"epoch"`: the gradients are then cleared once per epoch, before the batch loop, instead of once per batch.

**What should change:** the first epoch prints exactly the same row as the copy without `zero_grad()` in
step 8 (16.570), because nothing is cleared between its batches. Inside each epoch the gradients pile up again,
so the loss jumps around and ends far above 1.441. A line in the wrong loop raises no error either.

In [12]:
ZERO_GRAD_EVERY = "batch"  # <- change: try "epoch"
model, loss_fn, optimizer, train_loader, train_jobs, train_hours, val_jobs, val_hours = setup()
for epoch in range(1, EPOCHS + 1):
    model.train()
    if ZERO_GRAD_EVERY == "epoch":
        optimizer.zero_grad()
    for jobs, hours_batch in train_loader:
        predicted = model(jobs)
        loss = loss_fn(predicted, hours_batch)
        if ZERO_GRAD_EVERY == "batch":
            optimizer.zero_grad()
        loss.backward()
        optimizer.step()
    model.eval()
    with torch.no_grad():
        train_loss = loss_fn(model(train_jobs), train_hours)
        val_loss = loss_fn(model(val_jobs), val_hours)
        val_mae = (model(val_jobs) - val_hours).abs().mean()
    report(epoch, train_loss, val_loss, val_mae)

epoch  1  train loss    11.397  val loss    11.634  val off by   2.27 h
epoch  2  train loss     9.873  val loss    10.304  val off by   2.25 h
epoch  3  train loss     9.322  val loss     9.530  val off by   1.83 h
epoch  4  train loss     7.616  val loss     8.139  val off by   1.95 h
epoch  5  train loss     6.919  val loss     7.306  val off by   1.61 h
epoch  6  train loss     5.986  val loss     6.610  val off by   1.70 h


epoch  7  train loss     5.628  val loss     6.014  val off by   1.39 h
epoch  8  train loss     4.891  val loss     5.574  val off by   1.51 h
epoch  9  train loss     4.349  val loss     4.964  val off by   1.36 h
epoch 10  train loss     4.172  val loss     4.688  val off by   1.34 h
epoch 11  train loss     3.720  val loss     4.326  val off by   1.31 h
epoch 12  train loss     4.075  val loss     4.475  val off by   1.37 h
epoch 13  train loss     4.805  val loss     5.691  val off by   1.75 h
epoch 14  train loss     2.999  val loss     3.574  val off by   1.22 h
epoch 15  train loss     3.091  val loss     3.796  val off by   1.34 h
epoch 16  train loss     2.651  val loss     3.268  val off by   1.20 h
epoch 17  train loss     4.368  val loss     5.176  val off by   1.68 h
epoch 18  train loss     2.509  val loss     2.952  val off by   1.13 h
epoch 19  train loss     2.333  val loss     2.765  val off by   1.09 h
epoch 20  train loss     2.090  val loss     2.577  val off by  

epoch 28  train loss     1.438  val loss     1.735  val off by   0.88 h
epoch 29  train loss     1.409  val loss     1.721  val off by   0.90 h
epoch 30  train loss     1.441  val loss     1.753  val off by   0.92 h


## What you should see

- With 300 training jobs and `batch_size=32`, the loop makes 10 weight updates per epoch (300 after 30 epochs).
- The training loss starts near 11 in epoch 1 and ends below 2 by epoch 30, with a few bumps on the way.
- The validation loss stays a little above the training loss in every epoch.
- With `optimizer.step()` deleted, every epoch prints the same loss and the first layer's weights change by 0.
- With `loss.backward()` deleted, the loss is just as flat, but the gradient norm prints `None`.
- With `optimizer.zero_grad()` deleted, the loss falls for a few epochs, then swings and climbs.